## **Setup confirmation**

In [1]:
import boto3

bedrock = boto3.client("bedrock", region_name="us-east-1")
response = bedrock.list_foundation_models()
print(f"Available models: {len(response['modelSummaries'])}")

Available models: 131


## **Practice 1: First Call to Claude**

In [3]:
import boto3
import json

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")
model_id = "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0"


def invoke_claude(prompt, model_id):
    body = json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1000,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    response = bedrock_runtime.invoke_model(modelId=model_id, body=body)

    response_body = json.loads(response["body"].read())
    return response_body["content"][0]["text"]


# Ejemplo de uso
resultado = invoke_claude("Explica qué es AWS Bedrock en 3 párrafos", model_id)
print(resultado)

AWS Bedrock es una plataforma de computación en la nube desarrollada por Amazon Web Services que permite a empresas y desarrolladores acceder y utilizar modelos de inteligencia artificial generativa de forma sencilla y segura. Esta herramienta ofrece acceso a varios modelos de lenguaje de gran potencia, como los de Anthropic, AI21 Labs, Stability AI y Amazon, permitiendo a los usuarios crear aplicaciones inteligentes sin necesidad de gestionar la infraestructura subyacente.

La principal característica de Bedrock es su capacidad para proporcionar modelos de IA de última generación a través de una interfaz unificada y completamente administrada. Los usuarios pueden seleccionar, personalizar y integrar estos modelos en sus aplicaciones empresariales con facilidad, aprovechando capacidades como generación de texto, análisis de documentos, creación de imágenes y procesamiento de lenguaje natural. Además, la plataforma garantiza altos estándares de seguridad y privacidad de los datos.

Entr

## **Practice 2: Model Comparison**

In [ ]:
import json
import boto3

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

model_ids = [
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/global.amazon.nova-2-lite-v1:0",
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-haiku-20240307-v1:0",
]


def invoke_model(prompt, model_id):
    """Invoca diferentes modelos con el formato de request body apropiado"""

    # Detectar si es un modelo Claude o Amazon Nova
    is_claude = "anthropic.claude" in model_id
    is_nova = "amazon.nova" in model_id

    # Construir el body según el modelo
    if is_claude:
        body = json.dumps(
            {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 1000,
                "messages": [
                    {"role": "user", "content": prompt}  # Claude acepta string directo
                ],
            }
        )
    elif is_nova:
        body = json.dumps(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": [{"text": prompt}],  # Nova REQUIERE array de objetos
                    }
                ],
                "inferenceConfig": {"max_new_tokens": 1000},
            }
        )
    else:
        # Formato genérico para otros modelos
        body = json.dumps(
            {"max_tokens": 1000, "messages": [{"role": "user", "content": prompt}]}
        )

    response = bedrock_runtime.invoke_model(modelId=model_id, body=body)

    response_body = json.loads(response["body"].read())

    # Extraer el texto según el formato de respuesta
    if is_claude:
        return response_body["content"][0]["text"]
    elif is_nova:
        return response_body["output"]["message"]["content"][0]["text"]
    else:
        # Intentar extraer de forma genérica
        return response_body.get("content", [{}])[0].get("text", str(response_body))


# Prompt a usar
prompt = "Explica qué es AWS Bedrock en 3 párrafos"

# Iterar sobre cada modelo y mostrar resultados
print("=" * 80)
for model_id in model_ids:
    # Extraer nombre corto del modelo para mejor visualización
    model_name = model_id.split("/")[-1] if "/" in model_id else model_id

    print(f"\n🤖 MODEL: {model_name}")
    print(f"   Full ARN: {model_id}")
    print("-" * 80)
    try:
        resultado = invoke_model(prompt, model_id)
        print(resultado)
    except Exception as e:
        print(f"❌ Error: {e}")
        print(f"   Tipo: {type(e).__name__}")
    print("=" * 80)


🤖 MODEL: us.anthropic.claude-3-5-haiku-20241022-v1:0
   Full ARN: arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0
--------------------------------------------------------------------------------
AWS Bedrock es una plataforma de inteligencia artificial generativa de Amazon Web Services (AWS) que permite a desarrolladores y empresas acceder y utilizar diversos modelos de lenguaje de gran escala (LLM) de manera sencilla y segura. A diferencia de otras soluciones, Bedrock ofrece una interfaz unificada que facilita la integración de modelos de diferentes proveedores como Anthropic, AI21 Labs y Stability AI, sin necesidad de gestionar infraestructura compleja.

La plataforma destaca por su enfoque en la facilidad de uso y la seguridad. Permite a los usuarios seleccionar, personalizar y desplegar modelos de IA generativa mediante una API simple, con opciones de configuración flexibles y controles robustos de privacidad y acceso. Además, Be

## **Practice 3: Parameter Handling**

In [ ]:
import json
import boto3

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

# Usar solo el primer modelo de Anthropic
model_id = "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0"

# Lista de temperaturas a probar
temperatures = [0, 0.3, 0.7, 1.0, 1.5]

"""
PARÁMETROS DE SAMPLING EXPLICADOS:

Temperature: Acentúa o suaviza la distribución de logits. Valores bajos (0-0.3) hacen
que solo los tokens con mayor probabilidad sean seleccionados durante el sampling,
resultando en respuestas más determinísticas. Valores altos (0.7-1.5) suavizan la
distribución, permitiendo más variabilidad y creatividad.

Top_p (nucleus sampling): Durante el sampling, solo considera tokens cuyas probabilidades
acumuladas sumen menos que el valor de top_p. El resto de tokens queda descartado.
Ejemplo: top_p=0.9 considera solo los tokens que en conjunto suman 90% de probabilidad.

Top_k: En vez de usar la sumatoria de probabilidades, simplemente selecciona los top_k
tokens con mayor probabilidad y descarta el resto durante el sampling.

CUÁNDO USAR CADA UNO:
- Temperature baja (0-0.3): Tareas técnicas, código, respuestas consistentes
- Temperature media (0.5-0.7): Balance general, uso común
- Temperature alta (0.8-1.5): Tareas creativas, brainstorming, variación
- Top_p (0.9-0.95): Control más dinámico que top_k, recomendado por defecto
- Top_k (40-100): Control más estricto, útil cuando necesitas limitar vocabulario
"""


def invoke_claude_with_temperature(
    prompt, model_id, temperature, top_p=None, top_k=None
):
    """Invoca Claude con una temperatura específica y parámetros opcionales de sampling"""
    body_params = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 50,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}],
    }

    # Agregar top_p si se especifica
    if top_p is not None:
        body_params["top_p"] = top_p

    # Agregar top_k si se especifica
    if top_k is not None:
        body_params["top_k"] = top_k

    body = json.dumps(body_params)

    response = bedrock_runtime.invoke_model(modelId=model_id, body=body)

    response_body = json.loads(response["body"].read())
    return response_body["content"][0]["text"]


# Prompt a usar
prompt = "Explica qué es AWS Bedrock en 3 párrafos"

# Extraer nombre del modelo
model_name = model_id.split("/")[-1]

print("=" * 80)
print(f"🤖 MODELO: {model_name}")
print(f"📝 PROMPT: {prompt}")
print(f"🔁 EJECUTAR 5 VECES PARA OBSERVAR VARIABILIDAD")
print("=" * 80)

# Iterar sobre cada temperatura
for temp in temperatures:
    print(f"\n🌡️  TEMPERATURE: {temp}")
    print("=" * 80)

    # 1. Sin parámetros adicionales (solo temperature)
    print("\n📌 SIN PARÁMETROS ADICIONALES")
    print("-" * 80)
    try:
        resultado = invoke_claude_with_temperature(prompt, model_id, temp)
        print(resultado)
    except Exception as e:
        print(f"❌ Error: {e}")

    # 2. Con top_p
    print("\n📌 CON TOP_P = 0.9")
    print("-" * 80)
    try:
        resultado = invoke_claude_with_temperature(prompt, model_id, temp, top_p=0.9)
        print(resultado)
    except Exception as e:
        print(f"❌ Error: {e}")

    # 3. Con top_k
    print("\n📌 CON TOP_K = 50")
    print("-" * 80)
    try:
        resultado = invoke_claude_with_temperature(prompt, model_id, temp, top_k=50)
        print(resultado)
    except Exception as e:
        print(f"❌ Error: {e}")

    print("=" * 80)

🤖 MODELO: us.anthropic.claude-3-5-haiku-20241022-v1:0
📝 PROMPT: Explica qué es AWS Bedrock en 3 párrafos
🔁 EJECUTAR 5 VECES PARA OBSERVAR VARIABILIDAD

🌡️  TEMPERATURE: 0

📌 SIN PARÁMETROS ADICIONALES
--------------------------------------------------------------------------------
AWS Bedrock es un servicio de computación en la nube de Amazon diseñado para facilitar el desarrollo y la implementación de aplicaciones de inteligencia artificial generativa. Ofrece acceso a modelos de

📌 CON TOP_P = 0.9
--------------------------------------------------------------------------------
AWS Bedrock es un servicio de computación en la nube de Amazon diseñado para facilitar el desarrollo y la implementación de aplicaciones de inteligencia artificial generativa. Ofrece acceso a modelos de

📌 CON TOP_K = 50
--------------------------------------------------------------------------------
AWS Bedrock es un servicio de computación en la nube de Amazon diseñado para facilitar el desarrollo y la impleme

## **Practice 4: Response Streaming**

In [ ]:
import json
import boto3
from tqdm.auto import tqdm
from IPython.display import display, clear_output
import sys

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

model_id = "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0"


def stream_claude_response(prompt, model_id, max_tokens=500):
    """
    Stream de respuesta de Claude con contador de tokens en tiempo real
    """
    body = json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    # Iniciar streaming
    response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=model_id, body=body
    )

    # Variables para tracking
    token_count = 0
    full_text = ""

    # Barra de progreso
    pbar = tqdm(total=max_tokens, desc="🚀 Streaming", unit=" tokens")

    try:
        # Procesar el stream
        for event in response["body"]:
            chunk = json.loads(event["chunk"]["bytes"])

            # Verificar tipo de evento
            if chunk["type"] == "content_block_delta":
                # Extraer texto del delta
                if "delta" in chunk and "text" in chunk["delta"]:
                    text_chunk = chunk["delta"]["text"]
                    full_text += text_chunk
                    token_count += 1

                    # Actualizar barra
                    pbar.update(1)

                    # Limpiar output y mostrar actualizado
                    clear_output(wait=True)
                    print(f"🚀 Streaming: {token_count}/{max_tokens} tokens")
                    print(f"\n[Tokens: {token_count}]\n{full_text}")

            elif chunk["type"] == "message_stop":
                pbar.close()
                clear_output(wait=True)
                print(f"✅ Stream completado - {token_count} tokens")
                print(f"\n{full_text}")
                break

    except Exception as e:
        pbar.close()
        print(f"\n❌ Error durante streaming: {e}")
        return None, token_count

    return full_text, token_count


# Prompt a usar
prompt = "Explica qué es AWS Bedrock y menciona 3 servicios principales que integra"

print("=" * 100)
print(f"📝 PROMPT: {prompt}")
print("=" * 100)

# Ejecutar streaming
full_response, total_tokens = stream_claude_response(prompt, model_id, max_tokens=300)

print("\n" + "=" * 100)
print(f"📊 RESUMEN:")
print(f"   Total tokens generados: {total_tokens}")
print(f"   Longitud del texto: {len(full_response)} caracteres")
print("=" * 100)

✅ Stream completado - 75 tokens

AWS Bedrock es una plataforma de inteligencia artificial generativa de Amazon Web Services que permite a los desarrolladores y empresas crear y escalar aplicaciones de IA generativa de manera sencilla y segura.

Características principales:

1. Acceso a modelos de IA de múltiples proveedores
- Anthropic
- AI21 Labs
- Stability AI
- Amazon

3 Servicios principales que integra:

1. Claude AI (de Anthropic)
- Modelo de lenguaje avanzado
- Capacidad de procesamiento de texto
- Generación de contenido y análisis

2. Titan Text (modelo propio de Amazon)
- Generación de texto
- Resúmenes
- Traducción
- Clasificación de texto

3. Stable Diffusion (de Stability AI)
- Generación de imágenes
- Edición de imágenes
- Transformación visual mediante IA

Beneficios principales:
- Fácil integración
- Modelos de alta calidad
- Seguridad de datos
- Escalabilidad
- Flexibilidad

Principales casos de uso:
- Chatbots
- Generación de contenido
- Análisis de documentos

📊 RESU

## **Practice 6: Basic RAG with Embeddings**

In [12]:
!pip install langchain-community pypdf langchain-text-splitters sqlite-vec

In [3]:
# ============================================================================================================
# RAG SYSTEM - DOCUMENT INGESTION (ChromaDB Version)
# Ingesta de documentos PDF a vector store con embeddings de AWS Bedrock
# ============================================================================================================

import boto3
import os
from pathlib import Path
from langchain_aws import BedrockEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# ============================================================================================================
# CONFIGURATION
# ============================================================================================================
REGION = "us-east-1"
CHROMA_DB_DIR = "./chroma_db"
COLLECTION_NAME = "python_docs"
PDF_FILE = "styleguide _ Style guides for Google-originated open-source projects.pdf"

print("=" * 100)
print("🚀 STARTING RAG INGESTION PIPELINE (ChromaDB)")
print("=" * 100)

# ============================================================================================================
# STEP 1: LOAD EMBEDDINGS
# ============================================================================================================
print("\n📦 Step 1: Loading embedding model...")
embeddings = BedrockEmbeddings(
    region_name=REGION,
    model_id="amazon.titan-embed-text-v1"
)
print("✅ Embeddings loaded\n")

# ============================================================================================================
# STEP 2: LOAD PDF DOCUMENT
# ============================================================================================================
print("📄 Step 2: Loading PDF document...")
if not os.path.exists(PDF_FILE):
    raise FileNotFoundError(f"File not found: {PDF_FILE}")

loader = PyPDFLoader(file_path=PDF_FILE)
documents = loader.load()
print(f"✅ Loaded {len(documents)} pages\n")

# ============================================================================================================
# STEP 3: SPLIT INTO CHUNKS
# ============================================================================================================
print("✂️  Step 3: Splitting document into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks\n")

# Show sample chunk
print("📝 Sample chunk:")
print("-" * 100)
print(chunks[0].page_content[:300] + "...")
print(f"Metadata: {chunks[0].metadata}")
print("-" * 100 + "\n")

# ============================================================================================================
# STEP 4: CREATE VECTOR STORE WITH CHROMADB
# ============================================================================================================
print("🗄️  Step 4: Creating ChromaDB vector store (this may take a few minutes)...")

# Create Chroma DB from documents
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DB_DIR
)

print(f"✅ Vector store created with {len(chunks)} documents")
print(f"📁 Database location: {CHROMA_DB_DIR}\n")

# ============================================================================================================
# STEP 5: TEST SIMILARITY SEARCH
# ============================================================================================================
print("🔍 Step 5: Testing semantic search...")

test_queries = [
    "What are the main Python style guidelines?",
    "How should I format Python code?",
    "What are the naming conventions?"
]

for query in test_queries:
    print(f"\n🔎 Query: '{query}'")
    print("-" * 100)

    results = db.similarity_search(query, k=2)

    for i, doc in enumerate(results, 1):
        print(f"\n📄 Result {i} (relevance score):")
        print(f"{doc.page_content[:200]}...")
        print(f"Source: Page {doc.metadata.get('page', 'N/A')}")

    print("-" * 100)

# ============================================================================================================
# STEP 6: TEST SIMILARITY SEARCH WITH SCORES
# ============================================================================================================
print("\n📊 Step 6: Testing similarity search with scores...")
query = "Python naming conventions"
results_with_scores = db.similarity_search_with_score(query, k=3)

print(f"\n🔎 Query: '{query}'")
print("-" * 100)
for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n📄 Result {i} - Score: {score:.4f}")
    print(f"{doc.page_content[:150]}...")
    print(f"Page: {doc.metadata.get('page', 'N/A')}")
print("-" * 100)

# ============================================================================================================
# SUMMARY
# ============================================================================================================
print("\n" + "=" * 100)
print("✅ INGESTION COMPLETED SUCCESSFULLY")
print("=" * 100)
print(f"📁 Database directory: {CHROMA_DB_DIR}")
print(f"📦 Collection name: {COLLECTION_NAME}")
print(f"📊 Total chunks: {len(chunks)}")
print(f"📄 Original pages: {len(documents)}")
print(f"🔢 Embedding dimension: {len(embeddings.embed_query('test'))}")
print("=" * 100)

# ============================================================================================================
# SAVE REFERENCE FOR LATER USE
# ============================================================================================================
print("\n💾 Vector store 'db' available for queries")
print("💡 Usage examples:")
print("   - db.similarity_search('your query', k=3)")
print("   - db.similarity_search_with_score('your query', k=3)")
print("   - db.as_retriever(search_kwargs={'k': 5})")
print("\n✨ You can now use this in your multi-agent system!")


🚀 STARTING RAG INGESTION PIPELINE (ChromaDB)

📦 Step 1: Loading embedding model...
✅ Embeddings loaded

📄 Step 2: Loading PDF document...
✅ Loaded 58 pages

✂️  Step 3: Splitting document into chunks...
✅ Created 134 chunks

📝 Sample chunk:
----------------------------------------------------------------------------------------------------
styleguide
Google Python Style Guide
Table of Contents
1 Background
Python is the main dynamic language used at Google. This style guide is a list of dos and donʼts
for Python programs.
To help you format code correctly, weʼve created a settings file for Vim. For Emacs, the default
settings should b...
Metadata: {'producer': 'Skia/PDF m143', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36', 'creationdate': '2025-12-20T07:31:44+00:00', 'title': 'styleguide | Style guides for Google-originated open-source projects', 'moddate': '2025-12-20T07:31:44+00:00', 'source': 'style